# P07 SkyOps — Week 7: Gold Metrics

### ZENAIZ × BVRIT Hyderabad Data Engineering Internship
**Project:** SkyOps Airline Delay Command Center  
**Project ID:** P07  
**Week:** 7 — Gold Metrics, KPI Definitions, Aggregation and Validation  
**Technology:** Databricks + Spark SQL + Delta

## Week 7 objective

Turn the **Week 6 Trusted Silver** flight data into a dashboard-ready Gold metric table.

**Live build:** `gold_skyops_carrier_daily_metrics`  
**Gold grain:** one row per **reporting carrier + flight date**

The notebook follows the supplied Week 7 method:

```text
Business question
      ↓
KPI contract
      ↓
Grain + scope
      ↓
Trusted Silver inputs
      ↓
Safe enrichment join
      ↓
Aggregation
      ↓
Gold Delta table
      ↓
Key / measure / reconciliation validation
```

> **Boundary:** Gold reads Trusted Silver only. It does not recreate Week 5 transformations, rerun Week 6 DQ routing, or read Quarantine as KPI input.


## Before you run this notebook

1. Complete the Week 6 SkyOps notebook.
2. Confirm these Trusted Silver tables exist:
   - `trusted_silver_flights`
   - `trusted_silver_carriers`
3. Use the same catalog/schema where Week 6 wrote its tables.
4. Run cells top-to-bottom in Databricks.
5. Capture the required evidence after the validation cells.

The notebook deliberately does **not** hard-code expected row counts. All counts and KPI values must come from the current Databricks tables.


## 1. Week 7 outcome and KPI register

### Business questions

| KPI | Business question | Gold grain | Treatment |
|---|---|---|---|
| Daily flight volume | How many trusted flight records occurred for each carrier each day? | Carrier + day | **Build end-to-end** |
| Cancellation rate | What share of trusted flight records were cancelled? | Carrier + day | Build as Gold measure |
| Diversion rate | What share of trusted flight records were diverted? | Carrier + day | Build as Gold measure |
| 15-minute departure delay rate | What share of eligible non-cancelled/non-diverted flights had signed departure delay ≥ 15 minutes? | Carrier + day | Build as Gold measure |
| Average departure operational delay | What was the average nonnegative departure delay among eligible flights? | Carrier + day | Build as Gold measure |
| Average arrival operational delay | What was the average nonnegative arrival delay among eligible flights? | Carrier + day | Build as Gold measure |
| Delay-cause totals | Which delay causes contributed the most minutes? | Carrier + day | Design target for a later Gold table |

### Why these definitions?

The supplied project data dictionary distinguishes **signed source delay** from **nonnegative operational delay**. Week 6 also derives the governed 15-minute interpretation from signed delay and validates cancellation/diversion consistency.

Therefore:
- use `*_delay_minutes` for average operational-delay measures;
- use `departure_delay_signed_minutes >= 15` for the 15-minute departure-delay KPI;
- do not invent a separate source 15-minute flag;
- cancelled/diverted rows are excluded from the delay-rate and average-delay eligibility denominator.


## 2. KPI contracts — formulas, grain and scope

### 2.1 Daily flight volume

**Formula:** `COUNT(*)`  
**Grain:** `reporting_carrier + flight_date`  
**Eligible rows:** Trusted Silver flight rows with a usable `flight_date`  
**Outside scope:** rows with null `flight_date`  
**Measure:** `total_flights`

### 2.2 Cancellation rate

**Formula:** `cancelled_flights / total_flights`  
**Denominator:** all in-scope trusted flight rows  
**Measure:** `cancellation_rate`

### 2.3 Diversion rate

**Formula:** `diverted_flights / total_flights`  
**Denominator:** all in-scope trusted flight rows  
**Measure:** `diversion_rate`

### 2.4 15-minute departure delay rate

**Formula:** `departure_delay_15m_flights / eligible_departure_delay_flights`  
**Eligible denominator:** `cancelled_flag = 0`, `diverted_flag = 0`, and `departure_delay_signed_minutes IS NOT NULL`  
**Numerator:** eligible rows where `departure_delay_signed_minutes >= 15`  
**Measure:** `departure_delay_15m_rate`

### 2.5 Average departure operational delay

**Formula:** `AVG(departure_delay_minutes)`  
**Eligible rows:** non-cancelled, non-diverted trusted flights with non-null `departure_delay_minutes`  
**Measure:** `avg_departure_operational_delay_minutes`

### 2.6 Average arrival operational delay

**Formula:** `AVG(arrival_delay_minutes)`  
**Eligible rows:** non-cancelled, non-diverted trusted flights with non-null `arrival_delay_minutes`  
**Measure:** `avg_arrival_operational_delay_minutes`

> **Metric-grain rule:** every row in the live Gold table represents exactly one `reporting_carrier + metric_date`. Loan/flight-level identifiers are not carried into the aggregated output.


## 3. Confirm the Week 6 handoff

Week 7 must begin from Trusted Silver. If the Week 6 tables are missing, stop and complete Week 6 first.


In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;

In [ ]:
%sql
SHOW TABLES LIKE 'trusted_silver_*';

In [ ]:
%sql
SELECT
    'trusted_silver_flights' AS entity,
    COUNT(*) AS trusted_rows
FROM trusted_silver_flights
UNION ALL
SELECT
    'trusted_silver_carriers',
    COUNT(*)
FROM trusted_silver_carriers;

**Checkpoint:** the active catalog/schema must match the Week 6 handoff. The two Trusted Silver tables used by this notebook must exist before continuing.


## 4. Close the Week 6 physical handoff before Gold

Gold should be based only on Trusted records. The following check gives the Week 6 Candidate → Trusted + Quarantine context without rebuilding DQ logic.


In [ ]:
%sql
SELECT
    'flights' AS entity,
    (SELECT COUNT(*) FROM silver_flights_candidate) AS candidate_rows,
    (SELECT COUNT(*) FROM trusted_silver_flights) AS trusted_rows,
    (SELECT COUNT(*) FROM quarantine_flights) AS quarantine_rows,
    CASE
      WHEN (SELECT COUNT(*) FROM silver_flights_candidate)
         = (SELECT COUNT(*) FROM trusted_silver_flights)
         + (SELECT COUNT(*) FROM quarantine_flights)
      THEN 'PASS' ELSE 'CHECK'
    END AS reconciliation_status;

**Expected result:** `PASS`. This is a handoff check only; Week 7 does not alter Candidate, Trusted or Quarantine.


## 5. Inspect only the fields needed for the KPI contract

Gold should not copy every Silver column. The live build needs the flight date, carrier identity, cancellation/diversion flags and delay fields, plus carrier name for dashboard display.


In [ ]:
%sql
SELECT
    flight_id,
    flight_date,
    reporting_carrier,
    cancelled_flag,
    diverted_flag,
    departure_delay_signed_minutes,
    departure_delay_minutes,
    arrival_delay_signed_minutes,
    arrival_delay_minutes
FROM trusted_silver_flights
LIMIT 10;

In [ ]:
%sql
SELECT
    carrier_code,
    carrier_name,
    active_flag
FROM trusted_silver_carriers
LIMIT 10;

## 6. Profile Gold scope and KPI eligibility

Do not silently filter rows. Profile the different denominators used by the KPI formulas.

- `daily_volume_eligible`: usable `flight_date`
- `departure_delay_eligible`: non-cancelled + non-diverted + signed departure delay available
- `arrival_delay_eligible`: non-cancelled + non-diverted + operational arrival delay available


In [ ]:
%sql
SELECT
    COUNT(*) AS trusted_flight_rows,
    SUM(CASE WHEN flight_date IS NOT NULL THEN 1 ELSE 0 END) AS daily_volume_eligible,
    SUM(CASE WHEN flight_date IS NULL THEN 1 ELSE 0 END) AS outside_daily_volume_scope,
    SUM(CASE
          WHEN cancelled_flag = 0
           AND diverted_flag = 0
           AND departure_delay_signed_minutes IS NOT NULL
          THEN 1 ELSE 0
        END) AS departure_delay_eligible,
    SUM(CASE
          WHEN cancelled_flag = 0
           AND diverted_flag = 0
           AND arrival_delay_minutes IS NOT NULL
          THEN 1 ELSE 0
        END) AS arrival_delay_eligible
FROM trusted_silver_flights;

In [ ]:
%sql
WITH scope AS (
  SELECT
    COUNT(*) AS trusted_rows,
    SUM(CASE WHEN flight_date IS NOT NULL THEN 1 ELSE 0 END) AS eligible_rows,
    SUM(CASE WHEN flight_date IS NULL THEN 1 ELSE 0 END) AS outside_scope_rows
  FROM trusted_silver_flights
)
SELECT *,
       CASE
         WHEN trusted_rows = eligible_rows + outside_scope_rows
         THEN 'PASS' ELSE 'CHECK'
       END AS scope_reconciliation
FROM scope;

**Expected result:** `scope_reconciliation = PASS`. If there are outside-scope rows, keep them in Trusted Silver; they are simply not eligible for this daily Gold grain.


## 7. Prove the carrier lookup is safe

Week 6 validated carrier references. Week 7 reconfirms the lookup key before joining because a duplicate carrier row would multiply flight records and inflate every KPI.


In [ ]:
%sql
SELECT
    carrier_code,
    COUNT(*) AS occurrences
FROM trusted_silver_carriers
GROUP BY carrier_code
HAVING COUNT(*) > 1;

**Expected result:** zero returned groups.


## 8. Build a row-preserving enriched flight view

Use a `LEFT JOIN` first. A trusted flight should resolve to exactly one trusted carrier row. The join must preserve the flight grain before aggregation.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW skyops_daily_metric_input AS
SELECT
    f.flight_id,
    f.flight_date,
    f.reporting_carrier,
    c.carrier_code AS matched_carrier_code,
    c.carrier_name,
    f.cancelled_flag,
    f.diverted_flag,
    f.departure_delay_signed_minutes,
    f.departure_delay_minutes,
    f.arrival_delay_minutes
FROM trusted_silver_flights f
LEFT JOIN trusted_silver_carriers c
  ON f.reporting_carrier = c.carrier_code;

In [ ]:
%sql
WITH join_checks AS (
  SELECT
    (SELECT COUNT(*) FROM trusted_silver_flights) AS before_join_rows,
    (SELECT COUNT(*) FROM skyops_daily_metric_input) AS after_join_rows,
    (SELECT COUNT(*)
       FROM skyops_daily_metric_input
      WHERE matched_carrier_code IS NULL) AS unmatched_carrier_rows
)
SELECT *,
       CASE
         WHEN before_join_rows = after_join_rows
          AND unmatched_carrier_rows = 0
         THEN 'PASS' ELSE 'CHECK'
       END AS join_status
FROM join_checks;

In [ ]:
%sql
SELECT
    flight_id,
    COUNT(*) AS occurrences
FROM skyops_daily_metric_input
GROUP BY flight_id
HAVING COUNT(*) > 1;

**Expected result:** `join_status = PASS` and zero repeated `flight_id` values. Do not aggregate until both conditions pass.


## 9. Apply the approved scope and derive the Gold date

`metric_date` is an analytical field derived from the trusted `flight_date`. This is a Gold modelling transformation, not DQ repair.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW skyops_daily_metric_eligible AS
SELECT
    flight_id,
    CAST(flight_date AS DATE) AS metric_date,
    reporting_carrier,
    carrier_name,
    cancelled_flag,
    diverted_flag,
    departure_delay_signed_minutes,
    departure_delay_minutes,
    arrival_delay_minutes
FROM skyops_daily_metric_input
WHERE flight_date IS NOT NULL;

In [ ]:
%sql
SELECT
    COUNT(*) AS eligible_input_rows,
    SUM(CASE WHEN metric_date IS NULL THEN 1 ELSE 0 END) AS null_metric_dates
FROM skyops_daily_metric_eligible;

**Expected result:** `null_metric_dates = 0`. The eligible row count must come from the current Trusted Silver data.


## 10. Preview the Gold aggregation

The source grain is one row per trusted flight. The Gold grain becomes one row per carrier per date.

The expressions below keep KPI denominators explicit rather than relying on a single hidden filter.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW gold_skyops_carrier_daily_metrics_preview AS
SELECT
    reporting_carrier,
    MAX(carrier_name) AS carrier_name,
    metric_date,

    COUNT(*) AS total_flights,

    SUM(CASE WHEN cancelled_flag = 1 THEN 1 ELSE 0 END) AS cancelled_flights,
    SUM(CASE WHEN diverted_flag = 1 THEN 1 ELSE 0 END) AS diverted_flights,

    CAST(
      SUM(CASE WHEN cancelled_flag = 1 THEN 1 ELSE 0 END)
      AS DOUBLE
    ) / NULLIF(COUNT(*), 0) AS cancellation_rate,

    CAST(
      SUM(CASE WHEN diverted_flag = 1 THEN 1 ELSE 0 END)
      AS DOUBLE
    ) / NULLIF(COUNT(*), 0) AS diversion_rate,

    SUM(
      CASE
        WHEN cancelled_flag = 0
         AND diverted_flag = 0
         AND departure_delay_signed_minutes IS NOT NULL
        THEN 1 ELSE 0
      END
    ) AS eligible_departure_delay_flights,

    SUM(
      CASE
        WHEN cancelled_flag = 0
         AND diverted_flag = 0
         AND departure_delay_signed_minutes IS NOT NULL
         AND departure_delay_signed_minutes >= 15
        THEN 1 ELSE 0
      END
    ) AS departure_delay_15m_flights,

    CAST(
      SUM(
        CASE
          WHEN cancelled_flag = 0
           AND diverted_flag = 0
           AND departure_delay_signed_minutes IS NOT NULL
           AND departure_delay_signed_minutes >= 15
          THEN 1 ELSE 0
        END
      ) AS DOUBLE
    ) / NULLIF(
      SUM(
        CASE
          WHEN cancelled_flag = 0
           AND diverted_flag = 0
           AND departure_delay_signed_minutes IS NOT NULL
          THEN 1 ELSE 0
        END
      ), 0
    ) AS departure_delay_15m_rate,

    AVG(
      CASE
        WHEN cancelled_flag = 0
         AND diverted_flag = 0
         AND departure_delay_minutes IS NOT NULL
        THEN departure_delay_minutes
      END
    ) AS avg_departure_operational_delay_minutes,

    AVG(
      CASE
        WHEN cancelled_flag = 0
         AND diverted_flag = 0
         AND arrival_delay_minutes IS NOT NULL
        THEN arrival_delay_minutes
      END
    ) AS avg_arrival_operational_delay_minutes

FROM skyops_daily_metric_eligible
GROUP BY reporting_carrier, metric_date;

In [ ]:
%sql
SELECT *
FROM gold_skyops_carrier_daily_metrics_preview
ORDER BY metric_date, reporting_carrier
LIMIT 20;

## 11. Validate the preview grain and measures before writing Delta


In [ ]:
%sql
SELECT
    SUM(CASE WHEN reporting_carrier IS NULL THEN 1 ELSE 0 END) AS null_carrier_keys,
    SUM(CASE WHEN metric_date IS NULL THEN 1 ELSE 0 END) AS null_date_keys,
    SUM(CASE WHEN total_flights <= 0 THEN 1 ELSE 0 END) AS non_positive_total_flights,
    SUM(CASE WHEN cancelled_flights < 0 THEN 1 ELSE 0 END) AS negative_cancelled_counts,
    SUM(CASE WHEN diverted_flights < 0 THEN 1 ELSE 0 END) AS negative_diverted_counts,
    SUM(CASE WHEN cancellation_rate < 0 OR cancellation_rate > 1 THEN 1 ELSE 0 END) AS invalid_cancellation_rates,
    SUM(CASE WHEN diversion_rate < 0 OR diversion_rate > 1 THEN 1 ELSE 0 END) AS invalid_diversion_rates,
    SUM(CASE WHEN departure_delay_15m_rate < 0 OR departure_delay_15m_rate > 1 THEN 1 ELSE 0 END) AS invalid_delay_rates
FROM gold_skyops_carrier_daily_metrics_preview;

In [ ]:
%sql
SELECT
    reporting_carrier,
    metric_date,
    COUNT(*) AS occurrences
FROM gold_skyops_carrier_daily_metrics_preview
GROUP BY reporting_carrier, metric_date
HAVING COUNT(*) > 1;

**Expected result:** all invalid-count/rate checks are zero and the duplicate-key query returns no rows.


## 12. Create the Gold Delta table

`CREATE OR REPLACE TABLE` follows the controlled snapshot pattern used in the supplied Week 6 material.

The Gold table contains only dashboard-facing dimensions, KPI measures and technical Gold metadata. It does not expose flight-level identifiers.


In [ ]:
%sql
CREATE OR REPLACE TABLE gold_skyops_carrier_daily_metrics
USING DELTA
AS
SELECT
    reporting_carrier,
    carrier_name,
    metric_date,

    total_flights,
    cancelled_flights,
    diverted_flights,
    cancellation_rate,
    diversion_rate,

    eligible_departure_delay_flights,
    departure_delay_15m_flights,
    departure_delay_15m_rate,

    avg_departure_operational_delay_minutes,
    avg_arrival_operational_delay_minutes,

    current_timestamp() AS _gold_created_at,
    'skyops_gold_v1.0' AS _gold_schema_version
FROM gold_skyops_carrier_daily_metrics_preview;

## 13. Inspect the Gold table contract


In [ ]:
%sql
DESCRIBE TABLE gold_skyops_carrier_daily_metrics;

In [ ]:
%sql
SELECT
    reporting_carrier,
    carrier_name,
    metric_date,
    total_flights,
    cancelled_flights,
    diverted_flights,
    cancellation_rate,
    diversion_rate,
    eligible_departure_delay_flights,
    departure_delay_15m_flights,
    departure_delay_15m_rate,
    avg_departure_operational_delay_minutes,
    avg_arrival_operational_delay_minutes,
    _gold_created_at,
    _gold_schema_version
FROM gold_skyops_carrier_daily_metrics
ORDER BY metric_date, reporting_carrier
LIMIT 20;

## 14. Validate Gold keys and KPI measures


In [ ]:
%sql
SELECT
    COUNT(*) AS gold_rows,
    SUM(CASE WHEN reporting_carrier IS NULL THEN 1 ELSE 0 END) AS null_carrier_keys,
    SUM(CASE WHEN metric_date IS NULL THEN 1 ELSE 0 END) AS null_date_keys,
    SUM(CASE WHEN total_flights <= 0 THEN 1 ELSE 0 END) AS invalid_total_flights,
    SUM(CASE WHEN cancelled_flights < 0 THEN 1 ELSE 0 END) AS invalid_cancelled_flights,
    SUM(CASE WHEN diverted_flights < 0 THEN 1 ELSE 0 END) AS invalid_diverted_flights,
    SUM(CASE WHEN cancellation_rate < 0 OR cancellation_rate > 1 THEN 1 ELSE 0 END) AS invalid_cancellation_rate,
    SUM(CASE WHEN diversion_rate < 0 OR diversion_rate > 1 THEN 1 ELSE 0 END) AS invalid_diversion_rate,
    SUM(CASE WHEN departure_delay_15m_rate < 0 OR departure_delay_15m_rate > 1 THEN 1 ELSE 0 END) AS invalid_departure_delay_rate
FROM gold_skyops_carrier_daily_metrics;

In [ ]:
%sql
SELECT
    reporting_carrier,
    metric_date,
    COUNT(*) AS occurrences
FROM gold_skyops_carrier_daily_metrics
GROUP BY reporting_carrier, metric_date
HAVING COUNT(*) > 1;

## 15. Reconcile Gold measures to eligible Trusted Silver

Aggregation reduces row count, so **Gold row count must not be compared with flight row count**.

For the daily flight-volume KPI, the correct reconciliation is:

`SUM(Gold total_flights) = COUNT(eligible Trusted Silver flight rows)`

For the delay KPIs, the corresponding eligibility denominators are also reconciled below.


In [ ]:
%sql
WITH trusted_metrics AS (
  SELECT
    COUNT(*) AS eligible_daily_volume_rows,
    SUM(CASE
          WHEN cancelled_flag = 0
           AND diverted_flag = 0
           AND departure_delay_signed_minutes IS NOT NULL
          THEN 1 ELSE 0
        END) AS trusted_departure_delay_eligible,
    SUM(CASE
          WHEN cancelled_flag = 0
           AND diverted_flag = 0
           AND departure_delay_signed_minutes IS NOT NULL
           AND departure_delay_signed_minutes >= 15
          THEN 1 ELSE 0
        END) AS trusted_departure_delay_15m,
    SUM(CASE
          WHEN cancelled_flag = 0
           AND diverted_flag = 0
           AND arrival_delay_minutes IS NOT NULL
          THEN 1 ELSE 0
        END) AS trusted_arrival_delay_eligible
  FROM trusted_silver_flights
  WHERE flight_date IS NOT NULL
),
gold_metrics AS (
  SELECT
    COALESCE(SUM(total_flights), 0) AS gold_total_flights,
    COALESCE(SUM(eligible_departure_delay_flights), 0) AS gold_departure_delay_eligible,
    COALESCE(SUM(departure_delay_15m_flights), 0) AS gold_departure_delay_15m,
    COALESCE(SUM(
      CASE
        WHEN avg_arrival_operational_delay_minutes IS NOT NULL
        THEN 0 ELSE 0
      END
    ), 0) AS placeholder_measure
  FROM gold_skyops_carrier_daily_metrics
)
SELECT
  t.eligible_daily_volume_rows,
  g.gold_total_flights,
  t.trusted_departure_delay_eligible,
  g.gold_departure_delay_eligible,
  t.trusted_departure_delay_15m,
  g.gold_departure_delay_15m,
  CASE
    WHEN t.eligible_daily_volume_rows = g.gold_total_flights
     AND t.trusted_departure_delay_eligible = g.gold_departure_delay_eligible
     AND t.trusted_departure_delay_15m = g.gold_departure_delay_15m
    THEN 'PASS' ELSE 'CHECK'
  END AS count_reconciliation_status
FROM trusted_metrics t
CROSS JOIN gold_metrics g;

The average-delay measures are averages, so they should be validated with a weighted recomputation rather than by summing Gold averages. The next cell checks that the Gold averages reproduce the detailed Trusted Silver averages when weighted by their eligible row counts.


In [ ]:
%sql
WITH trusted_daily AS (
  SELECT
    reporting_carrier,
    CAST(flight_date AS DATE) AS metric_date,
    SUM(CASE
          WHEN cancelled_flag = 0
           AND diverted_flag = 0
           AND departure_delay_minutes IS NOT NULL
          THEN departure_delay_minutes ELSE 0
        END) AS departure_delay_sum,
    SUM(CASE
          WHEN cancelled_flag = 0
           AND diverted_flag = 0
           AND arrival_delay_minutes IS NOT NULL
          THEN arrival_delay_minutes ELSE 0
        END) AS arrival_delay_sum,
    SUM(CASE
          WHEN cancelled_flag = 0
           AND diverted_flag = 0
           AND departure_delay_minutes IS NOT NULL
          THEN 1 ELSE 0
        END) AS departure_delay_n,
    SUM(CASE
          WHEN cancelled_flag = 0
           AND diverted_flag = 0
           AND arrival_delay_minutes IS NOT NULL
          THEN 1 ELSE 0
        END) AS arrival_delay_n
  FROM trusted_silver_flights
  WHERE flight_date IS NOT NULL
  GROUP BY reporting_carrier, CAST(flight_date AS DATE)
),
gold_recomputed AS (
  SELECT
    SUM(
      CASE WHEN g.avg_departure_operational_delay_minutes IS NOT NULL
           THEN g.avg_departure_operational_delay_minutes * t.departure_delay_n
           ELSE 0 END
    ) AS gold_departure_delay_sum,
    SUM(
      CASE WHEN g.avg_arrival_operational_delay_minutes IS NOT NULL
           THEN g.avg_arrival_operational_delay_minutes * t.arrival_delay_n
           ELSE 0 END
    ) AS gold_arrival_delay_sum,
    SUM(t.departure_delay_n) AS departure_delay_n,
    SUM(t.arrival_delay_n) AS arrival_delay_n,
    SUM(t.departure_delay_sum) AS trusted_departure_delay_sum,
    SUM(t.arrival_delay_sum) AS trusted_arrival_delay_sum
  FROM trusted_daily t
  INNER JOIN gold_skyops_carrier_daily_metrics g
    ON t.reporting_carrier = g.reporting_carrier
   AND t.metric_date = g.metric_date
)
SELECT
  trusted_departure_delay_sum,
  gold_departure_delay_sum,
  trusted_arrival_delay_sum,
  gold_arrival_delay_sum,
  CASE
    WHEN ABS(trusted_departure_delay_sum - gold_departure_delay_sum) < 0.000001
     AND ABS(trusted_arrival_delay_sum - gold_arrival_delay_sum) < 0.000001
    THEN 'PASS' ELSE 'CHECK'
  END AS average_measure_reconciliation
FROM gold_recomputed;

**Expected result:** `count_reconciliation_status = PASS` and `average_measure_reconciliation = PASS` (subject only to floating-point representation). A `CHECK` means the scope, join or aggregation logic must be investigated before submission.


## 16. Controlled repeat-run proof

The business rows should remain identical when the Gold build is rerun against the same Trusted Silver snapshot. Technical timestamps may change.


In [ ]:
%sql
CREATE OR REPLACE TABLE gold_skyops_carrier_daily_metrics_rerun_baseline
USING DELTA
AS
SELECT
    reporting_carrier,
    carrier_name,
    metric_date,
    total_flights,
    cancelled_flights,
    diverted_flights,
    cancellation_rate,
    diversion_rate,
    eligible_departure_delay_flights,
    departure_delay_15m_flights,
    departure_delay_15m_rate,
    avg_departure_operational_delay_minutes,
    avg_arrival_operational_delay_minutes
FROM gold_skyops_carrier_daily_metrics;

**Action:** rerun the build sections that create `skyops_daily_metric_input`, `skyops_daily_metric_eligible`, `gold_skyops_carrier_daily_metrics_preview`, and `gold_skyops_carrier_daily_metrics`. Then run the comparison below.


In [ ]:
%sql
WITH baseline_minus_current AS (
  SELECT *
  FROM gold_skyops_carrier_daily_metrics_rerun_baseline
  EXCEPT
  SELECT
    reporting_carrier,
    carrier_name,
    metric_date,
    total_flights,
    cancelled_flights,
    diverted_flights,
    cancellation_rate,
    diversion_rate,
    eligible_departure_delay_flights,
    departure_delay_15m_flights,
    departure_delay_15m_rate,
    avg_departure_operational_delay_minutes,
    avg_arrival_operational_delay_minutes
  FROM gold_skyops_carrier_daily_metrics
),
current_minus_baseline AS (
  SELECT
    reporting_carrier,
    carrier_name,
    metric_date,
    total_flights,
    cancelled_flights,
    diverted_flights,
    cancellation_rate,
    diversion_rate,
    eligible_departure_delay_flights,
    departure_delay_15m_flights,
    departure_delay_15m_rate,
    avg_departure_operational_delay_minutes,
    avg_arrival_operational_delay_minutes
  FROM gold_skyops_carrier_daily_metrics
  EXCEPT
  SELECT *
  FROM gold_skyops_carrier_daily_metrics_rerun_baseline
),
differences AS (
  SELECT * FROM baseline_minus_current
  UNION ALL
  SELECT * FROM current_minus_baseline
)
SELECT COUNT(*) AS changed_business_rows
FROM differences;

In [ ]:
%sql
DROP TABLE gold_skyops_carrier_daily_metrics_rerun_baseline;

**Expected result:** `changed_business_rows = 0`.


## 17. Final dashboard-ready evidence

Use the next outputs for the required Week 7 screenshots.

### Evidence 1 — `week07_gold_metrics.png`

Capture:
- Gold table schema / contract
- sample rows
- carrier + date grain
- KPI measures

### Evidence 2 — `week07_gold_validation.png`

Capture:
- Gold key/measure validation
- count reconciliation
- average-measure reconciliation
- repeat-run result

Do not type expected counts into the notebook. Capture the actual Databricks results.


In [ ]:
%sql
SELECT
    reporting_carrier,
    carrier_name,
    metric_date,
    total_flights,
    cancelled_flights,
    diverted_flights,
    cancellation_rate,
    diversion_rate,
    departure_delay_15m_flights,
    departure_delay_15m_rate,
    avg_departure_operational_delay_minutes,
    avg_arrival_operational_delay_minutes
FROM gold_skyops_carrier_daily_metrics
ORDER BY metric_date, reporting_carrier
LIMIT 25;

In [ ]:
%sql
SELECT
    'gold_skyops_carrier_daily_metrics' AS gold_table,
    COUNT(*) AS gold_rows,
    COUNT(DISTINCT CONCAT_WS('|', reporting_carrier, CAST(metric_date AS STRING))) AS distinct_gold_keys,
    CASE
      WHEN COUNT(*) =
           COUNT(DISTINCT CONCAT_WS('|', reporting_carrier, CAST(metric_date AS STRING)))
      THEN 'PASS' ELSE 'CHECK'
    END AS grain_status
FROM gold_skyops_carrier_daily_metrics;

## 18. Gold tables planned for later adaptation

Only the carrier-day table above is the complete Week 7 live build.

The following are design targets for later iterations, not fabricated completed outputs:

1. **`gold_skyops_airport_daily_metrics`** — one row per airport + day, with arrivals/departures and delay KPIs.
2. **`gold_skyops_route_daily_metrics`** — one row per directed route + day, with volume, cancellation/diversion and delay KPIs.
3. **`gold_skyops_delay_cause_daily_metrics`** — one row per carrier + day + cause type, sourced from Trusted Silver delay components after aggregation at the correct cause grain.

Delay-cause data must be aggregated at its own physical grain before being joined to flight-level metrics; this prevents one flight from multiplying Gold KPI rows, matching the Week 6 cause-grain rule.


## 19. GitHub updates and evidence checklist

The Week 7 sprint sheet calls for:

- `notebooks/05_gold_aggregations.ipynb` — **this notebook**
- `docs/gold_metrics_definition.md` — copy the KPI register/contracts from this notebook and keep the formulas/grain/scope synchronized
- `data_sample/gold_exports/` — save a small Gold output only if required by the project workflow
- `weekly_logs/week07_log.md` — record the build, validation results and AI Transparency Note
- Evidence:
  - `week07_gold_metrics.png`
  - `week07_gold_validation.png`

### Suggested commit messages

- `create gold carrier daily metric table`
- `document skyops gold metric definitions`
- `add week 7 gold validation evidence`
- `update week 7 log and AI transparency note`


## 20. Week 7 acceptance checklist

- [ ] Week 6 Trusted Silver handoff is present and reconciles.
- [ ] Gold reads Trusted Silver only.
- [ ] KPI formulas are explicitly documented.
- [ ] Gold grain is explicit: carrier + date.
- [ ] Scope/eligibility counts are shown before filtering.
- [ ] Carrier lookup is unique and join-preserving.
- [ ] Gold table is dashboard-ready and contains no flight-level identifier.
- [ ] Gold key has no duplicates.
- [ ] KPI rates stay between 0 and 1.
- [ ] Gold volume and delay-count measures reconcile to eligible Trusted Silver.
- [ ] Average-delay measures reconcile using weighted detail totals.
- [ ] Repeat-run business rows are unchanged.
- [ ] Evidence screenshots are captured.
- [ ] GitHub docs and Week 7 log are updated.


## AI Transparency Note

This Week 7 notebook was generated by adapting the supplied **Week 7 Gold-table learning pattern** to the supplied **P07 SkyOps Week 6 Trusted Silver design**, project data dictionary and Week 7 sprint sheet.

The supplied project materials do not provide a separate SkyOps Week 7 KPI register. Therefore this notebook makes the reporting definitions explicit rather than hiding them in SQL. The definitions use only fields and delay semantics already present in the supplied project materials. In particular, no independent source 15-minute flag or new DQ rule is invented.

Before submission, execute the notebook in the team's Databricks workspace and verify the actual table existence, schema, row counts, KPI values, reconciliation results and repeat-run proof.
